# 09 — Eksekusi terverifikasi
Data kompetisi asli; output di bawah direkam dari eksekusi 14 Oktober 2026. Jika checkpoint eksternal tidak tersedia, bagian historis berstatus SKIPPED. Training smoke bukan model 010.

In [1]:
from pathlib import Path
from zipfile import ZipFile
import io, hashlib, os
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, confusion_matrix
root = Path.cwd()
if not (root / "data").exists(): root = root.parent
data = root / "data/structural-damage-classification.zip"
final = root / "submissions/010_Constantine(10).csv"
checkpoint = Path(os.getenv("DAMAGE_CHECKPOINT_ZIP", str(root / "data/Structural_Damage_NonNeural_Rebuild_Checkpoint.zip")))
with ZipFile(data) as z:
    entries = {Path(k).name: k for k in z.namelist()}
    original = {k: z.read(entries[k]) for k in ["train.csv","test.csv","sample_submission.csv"]}
assert hashlib.sha256(original["train.csv"]).hexdigest() == "231c9bf0430faa9c1d2281b4edbcd9ce0d7a863c6ff67042a8fbc992469d37bb"
assert hashlib.sha256(original["test.csv"]).hexdigest() == "d732c3070b878ebc1346907d50e6a6a9db0eadf0790631b37c03d3ba6c5c"
assert hashlib.sha256(final.read_bytes()).hexdigest() == "2764f5f59a3924b982a72584edb4facdd8aeabde16519a8ce0e04e45975d0767"
train = pd.read_csv(io.BytesIO(original["train.csv"]),dtype={"building_id":str})
test = pd.read_csv(io.BytesIO(original["test.csv"]),dtype={"building_id":str})
sample = pd.read_csv(io.BytesIO(original["sample_submission.csv"]),dtype={"building_id":str})
submission = pd.read_csv(final,dtype={"building_id":str})
assert len(train)==533465 and len(test)==len(sample)==len(submission)==228629
assert submission.building_id.equals(sample.building_id) and submission.building_id.equals(test.building_id)
assert submission.damage_grade.isin(range(1,6)).all()
print("Integrity PASS | train=533465 | test=228629 | SHA-256 submission010 verified")

Integrity PASS | train=533465 | test=228629 | SHA-256 submission010 verified


In [2]:
if checkpoint.exists():
    with ZipFile(checkpoint) as z:
        for version in ("v6","v7"):
            with np.load(io.BytesIO(z.read(f"research_{version}/confirmation/pooled.npz"))) as cached:
                ids, ytrue, ypred = cached["indices"], cached["truth"]+1, cached["prediction"]+1
            np.testing.assert_array_equal(ytrue,train.damage_grade.iloc[ids].astype(str).str.extract(r"([1-5])$")[0].astype(int).to_numpy())
            print(version, "N=",len(ytrue),"Macro F1=",round(f1_score(ytrue,ypred,average="macro"),9))
        native = pd.read_csv(io.BytesIO(z.read("research_v7/full_refit/native_replay_submission.csv")),dtype={"building_id":str})
        assert native.building_id.equals(submission.building_id)
        difference = np.count_nonzero(native.damage_grade.to_numpy()!=submission.damage_grade.to_numpy())
        assert difference == 439
        print("Native v7 versus submission010 differing rows:", difference)
else:
    print("Historical checkpoint unavailable: SKIPPED (not passed)")

v6 N= 60000 Macro F1= 0.60079567
v7 N= 60000 Macro F1= 0.601198136
Native v7 versus submission010 differing rows: 439


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from lightgbm import LGBMClassifier
import joblib
X=train.drop(columns=["damage_grade","building_id"])
y=train.damage_grade.astype(str).str.extract(r"([1-5])$")[0].astype(int).to_numpy()
i,j=train_test_split(np.arange(len(y)),train_size=9600,test_size=2400,stratify=y,random_state=20261014)
numeric=X.select_dtypes("number").columns.tolist()
categorical=[col for col in X if col not in numeric]
preprocess=ColumnTransformer([
 ("numeric",SimpleImputer(strategy="median"),numeric),
 ("categorical",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("encoder",OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1))]),categorical)])
pipe=Pipeline([("columns",preprocess),("model",LGBMClassifier(n_estimators=85,num_leaves=15,learning_rate=.08,random_state=20261014,n_jobs=2,verbosity=-1))])
pipe.fit(X.iloc[i],y[i])
pred=pipe.predict(X.iloc[j])
output=root/"artifacts/reproduction_evidence"
output.mkdir(parents=True,exist_ok=True)
joblib.dump(pipe,output/"smoke_pipeline.joblib")
np.testing.assert_array_equal(pred,joblib.load(output/"smoke_pipeline.joblib").predict(X.iloc[j]))
print("Smoke fit/reload PASS | Macro F1",round(f1_score(y[j],pred,average="macro"),6))

Smoke fit/reload PASS | Macro F1 0.40474


Submission 010 intact; native v7 differs by 439 rows. Model final 010 is NOT proven reproducible byte-for-byte. No neural networks or hidden test labels.